In [1]:
recording_dir = 'recordings/Recorded_Demo'
h5filePath = recording_dir + '/frames.h5'

In [10]:
import tempfile
import h5py
import os
import shutil
import pyrealsense2 as rs
import numpy as np
from gemini_oop_object_detection import ObjectDetector
from gemini_constant_api_key import GEMINI_API_KEY
from utils import base64_to_csv_path, create_zip_archive, deproject_pixel_to_point, get_base64_encoded_hamer_response, get_intrinsics
from PIL import Image

X = np.array([
    [0.068, -0.986, 0.152, -0.108],
    [0.998, 0.065, -0.023, 0.0],
    [0.013, 0.153, 0.988, -0.044],
    [0.0, 0.0, 0.0, 1.0]
])

Y = np.array([
    [-0.47, 0.587, -0.659, 0.73929],
    [0.877, 0.392, -0.276, -0.16997],
    [0.096, -0.708, -0.7, 0.86356],
    [0.0, 0.0, 0.0, 1.0]
])

def collection_d(recording_dir, start_time, end_time, fps=10):
    
    """Get the frames corresponding to each action in the ELLM response."""
    with h5py.File(f"{recording_dir}/frames.h5", 'r') as h5_file:
        timestamps = h5_file['timestamps'][:]
        frame_times = timestamps[:, 0] - timestamps[0, 0]
        length_of_color= h5_file['color_frames'].shape[0]
        print("SHAPE OF COLOR FRAMES",length_of_color)
        time_duration = length_of_color/fps
        
        closest_start = start_time*fps
        closest_end = end_time*fps

        # Now based on the start frame number extract thecolor frame names.
        frame_paths = []
        depth_paths = []
        for frame_number in range(int(closest_start), int(closest_end)):
            depth_paths.append(f"{recording_dir}/depth_images_data_collection/image_{frame_number}.npy")
            frame_paths.append(f"{recording_dir}/rgb_images_data_collection/image_{frame_number}.jpg")
        
        # Create temporary directories for RGB and depth images
        temp_rgb_dir = os.path.join(tempfile.gettempdir(), 'temp_rgb')
        temp_depth_dir = os.path.join(tempfile.gettempdir(), 'temp_depth')
        os.makedirs(temp_rgb_dir, exist_ok=True)
        os.makedirs(temp_depth_dir, exist_ok=True)

        # Copy frames to temporary directories
        for i, (rgb_path, depth_path) in enumerate(zip(frame_paths, depth_paths)):
            shutil.copy(rgb_path, os.path.join(temp_rgb_dir, f'image_{i}.jpg'))
            shutil.copy(depth_path, os.path.join(temp_depth_dir, f'image_{i}.npy'))

        # Create zip archives for RGB and depth images
        rgb_zip_path = os.path.join(recording_dir, 'rgb_images_collection.zip')
        depth_zip_path = os.path.join(recording_dir, 'depth_images_collection.zip')
        create_zip_archive(temp_rgb_dir, rgb_zip_path)
        create_zip_archive(temp_depth_dir, depth_zip_path)

        # Hit the Hamer API with the newly generated zips
        response_encoded = get_base64_encoded_hamer_response(rgb_zip_path, depth_zip_path)
        if response_encoded:
            csv_path = base64_to_csv_path(response_encoded, f'{recording_dir}/predictions_hamer.csv')
        return csv_path

def transform_coordinates(x, y, z):
    """Transforms coordinates from input space to cobot base."""
    B = np.eye(4)
    B[:3, 3] = [x / 1000, y / 1000, z / 1000]  # Convert to meters
    A = Y @ B @ np.linalg.inv(X)
    transformed_x, transformed_y, transformed_z = A[:3, 3] * 1000  # Convert back to mm
    return transformed_x, transformed_y, transformed_z

def initial_frame_centers(recording_dir,classes):
    """
    This functions passes the 0th frame to gemini API and the API returns the x,y center point values of the objects.
    These points are then passed throug camera transformations to extract z from the corresponding depth frame and doing coodinate transformation
    """
    
    gemini_model = ObjectDetector(api_key=GEMINI_API_KEY,recording_dir=recording_dir)
    object_centers = {}
    rgb_image_path = f"{recording_dir}/rgb_images_data_collection/image_10.jpg"
    depth_image_path = f"{recording_dir}/depth_images_data_collection/image_10.npy"
    rgb_image = Image.open(rgb_image_path)
    depth_image = np.load(depth_image_path)
    intrinsics = rs.intrinsics()
    intrinsics.width = 640
    intrinsics.height = 480
    intrinsics.ppx = 329.1317443847656
    intrinsics.ppy = 240.29669189453125
    intrinsics.fx = 611.084594726562
    intrinsics.fy = 609.7639770507812
    intrinsics.model = rs.distortion.inverse_brown_conrady
    intrinsics.coeffs = [0, 0, 0, 0, 0]

    for name in classes:
        center_x, center_y, box, confidence = gemini_model.get_object_center(rgb_image, name)
        point_3d = deproject_pixel_to_point(depth_image, (center_x, center_y), intrinsics)
        print(point_3d)
        transformed_point_x,transformed_point_y,transformed_point_z = transform_coordinates(point_3d[0],point_3d[1],point_3d[2])
        transformed_point = [transformed_point_x,transformed_point_y,transformed_point_z]
        object_centers[name] = transformed_point
    
    return object_centers

In [8]:
ellm_response = {
    "overall_task_name": "picking up and placing a can",
    "objects": [
        "can",
        "small black box"
    ],
    "picking up": [
        {
            "start_time": "00:02",
            "end_time": "00:05",
            "object_name": "can",
            "notes": "Human reaches for and grasps the can.",
            "action_timestamp": "00:03"
        }
    ],
    "placing": [
        {
            "start_time": "00:05",
            "end_time": "00:07",
            "object_name": "can",
            "notes": "Human moves the can and places it down on the table.",
            "action_timestamp": "00:07"
        }
    ]
}
csv_path = collection_d(recording_dir=recording_dir, start_time=2, end_time=5, fps=10)

In [ ]:
input_dict = initial_frame_centers(recording_dir,["red soda can","white glass cup"])

Video properties:
- Duration: 7.60 seconds
- Frame count: 76
- FPS: 10.0
```json
{'red soda can': [226, 463, 351, 511]}
```
(640, 480)
[ -39.87837982 -225.47537231 1344.        ]
```json
{'white glass cup': [236, 592, 339, 654], 'red marker': [622, 13, 710, 70], 'green marker': [595, 25, 679, 81], 'person': [0, 131, 347, 503], 'robot arm': [101, 728, 999, 999], 'can': [228, 460, 354, 513]}
```
(640, 480)
[ 163.18728638 -242.92286682 1448.        ]


{'red soda can': [np.float64(-361.73874300785906),
  np.float64(-712.9976068394739),
  np.float64(108.03853293479997)],
 'white glass cup': [np.float64(-535.9572853967749),
  np.float64(-570.4524354298668),
  np.float64(67.085663000718)]}

In [18]:
import csv
import numpy as np

def post_process_csv(input_csv_path, output_csv_path, input_dict):
    with open(input_csv_path, mode='r') as infile, open(output_csv_path, mode='w', newline='') as outfile:
        reader = csv.DictReader(infile)
        fieldnames = ['Input_0_x', 'Input_0_y', 'Input_0_z', 'Input_1_x', 'Input_1_y', 'Input_1_z']
        
        for i in range(81):
            fieldnames.extend([f'P{i}_X', f'P{i}_Y', f'P{i}_Z', f'P{i}_C'])
        
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        
        row_data = {field: 0 for field in fieldnames}
        
        # Set input values from the dictionary
        row_data['Input_0_x'], row_data['Input_0_y'], row_data['Input_0_z'] = input_dict['white glass cup']
        row_data['Input_1_x'], row_data['Input_1_y'], row_data['Input_1_z'] = input_dict['red soda can']
        
        for i, row in enumerate(reader):
            row_data[f'P{i}_X'] = row['X']
            row_data[f'P{i}_Y'] = row['Y']
            row_data[f'P{i}_Z'] = row['Z']
            row_data[f'P{i}_C'] = row['C']
        
        writer.writerow(row_data)

# Usage
input_csv_path = 'c:/Users/Rushiil Bhatnagar/Downloads/object_detection/object_detection/recordings/20250108_172916/predictions_hamer.csv'
output_csv_path = 'c:/Users/Rushiil Bhatnagar/Downloads/object_detection/object_detection/recordings/20250108_172916/processed_predictions_hamer.csv'
input_dict = {
    'red soda can': [np.float64(-361.73874300785906), np.float64(-712.9976068394739), np.float64(108.03853293479997)],
    'white glass cup': [np.float64(-535.9572853967749), np.float64(-570.4524354298668), np.float64(67.085663000718)]
}
post_process_csv(input_csv_path, output_csv_path, input_dict)

In [19]:
import pandas as pd
data = pd.read_csv("recordings/20250108_172916/processed_predictions_hamer.csv")
data.head()

,Input_0_x,Input_0_y,Input_0_z,Input_1_x,Input_1_y,Input_1_z,P0_X,P0_Y,P0_Z,P0_C,...,P78_Z,P78_C,P79_X,P79_Y,P79_Z,P79_C,P80_X,P80_Y,P80_Z,P80_C
0,-535.957285,-570.452435,67.085663,-361.738743,-712.997607,108.038533,357.1525,-356.52505,275.32358,379.27292,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
import json
from object_detection.vdeo_analysis_ellm_sudio import VideoAnalyzer


def combined_pipeline(
    recording_dir,
    start_time,
    end_time,
    fps,
    classes,
    input_csv_path,
    output_csv_path,
    gemini_api_key,
    ellm_response=None
):
    """
    A single function pipeline that:
    1) Extracts frames and calls the Hamer API for predictions (collection_d).
    2) Gets initial object centers from the Gemini API (initial_frame_centers).
    3) Post-processes the CSV to incorporate 3D points (post_process_csv).

    :param recording_dir: The path to your main recording directory.
    :param start_time: Start time (in seconds).
    :param end_time: End time (in seconds).
    :param fps: Frames per second (integer).
    :param classes: List of class names (strings) to detect in initial_frame_centers.
    :param input_csv_path: Path to the predictions CSV (output of Hamer API).
    :param output_csv_path: Where to write your processed CSV.
    :param gemini_api_key: API key for your Gemini object detection service.
    :param ellm_response: A dictionary that can be used if you want to handle 
                          multiple actions/time stamps from LLM (optional).
    :return: Returns the path to the processed CSV.
    """

    import tempfile
    import h5py
    import os
    import shutil
    import pyrealsense2 as rs
    import numpy as np
    from PIL import Image
    import csv

    # ===================
    # Helper Functions
    # ===================

    # -- Example placeholder imports or local function references you might have:
    from gemini_oop_object_detection import ObjectDetector
    from gemini_constant_api_key import GEMINI_API_KEY
    from utils import (base64_to_csv_path, create_zip_archive, 
                       deproject_pixel_to_point, get_base64_encoded_hamer_response, 
                       get_intrinsics)

    def create_zip_archive(folder_path, output_zip_path):
        """Example local definition or import from your 'utils' module."""
        import zipfile
        with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(folder_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, folder_path)
                    zipf.write(file_path, arcname)

    def base64_to_csv_path(encoded_string, csv_path):
        """Example placeholder for your real base64-to-CSV logic."""
        import base64
        import csv
        import io
        
        decoded = base64.b64decode(encoded_string)
        with io.StringIO(decoded.decode('utf-8')) as s, open(csv_path, 'w', newline='') as out_csv:
            reader = csv.reader(s)
            writer = csv.writer(out_csv)
            for row in reader:
                writer.writerow(row)
        return csv_path

    def get_base64_encoded_hamer_response(rgb_zip_path, depth_zip_path):
        """Example placeholder to simulate an API call returning a base64-encoded CSV."""
        # This is where you'd integrate your actual network request code
        # For now, let's just return a dummy base64-encoded CSV of random data
        import base64
        dummy_csv = "X,Y,Z,C\n1,2,3,0.8\n4,5,6,0.9\n"  # Example content
        return base64.b64encode(dummy_csv.encode()).decode('utf-8')

    def deproject_pixel_to_point(depth_image, pixel, intrinsics):
        """
        Example placeholder for your real deprojection logic using pyrealsense2.
        Convert a pixel (x,y) and the depth at that pixel to 3D coordinates.
        """
        x, y = pixel
        depth_value = depth_image[int(y), int(x)] if (
            0 <= int(y) < depth_image.shape[0] and 0 <= int(x) < depth_image.shape[1]
        ) else 0

        if depth_value == 0:
            # Fallback or handle invalid depth
            return [0, 0, 0]

        # Use realsense2 intrinsics to deproject
        camera_coordinates = rs.rs2_deproject_pixel_to_point(intrinsics, [x, y], depth_value)
        return camera_coordinates

    # ===============================
    # 1) collection_d implementation
    # ===============================
    
    def collection_d(recording_dir, start_time, end_time, fps=10):
        """
        Get the frames corresponding to each action in the ELLM response, 
        call the Hamer API, and return path to CSV predictions.
        """
        h5_path = f"{recording_dir}/frames.h5"
        
        # Safety check
        if not os.path.exists(h5_path):
            raise FileNotFoundError(f"H5 file not found at {h5_path}")
        
        with h5py.File(h5_path, 'r') as h5_file:
            timestamps = h5_file['timestamps'][:]
            length_of_color = h5_file['color_frames'].shape[0]
            print("SHAPE OF COLOR FRAMES:", length_of_color)
            
            # Identify the frame indices we want
            closest_start = int(start_time * fps)
            closest_end = int(end_time * fps)

            # Derive the file paths for color and depth images
            frame_paths = [
                f"{recording_dir}/rgb_images_data_collection/image_{frame_num}.jpg"
                for frame_num in range(closest_start, closest_end)
            ]
            depth_paths = [
                f"{recording_dir}/depth_images_data_collection/image_{frame_num}.npy"
                for frame_num in range(closest_start, closest_end)
            ]

            # Temporary directories for copying frames
            temp_rgb_dir = os.path.join(tempfile.gettempdir(), 'temp_rgb')
            temp_depth_dir = os.path.join(tempfile.gettempdir(), 'temp_depth')
            os.makedirs(temp_rgb_dir, exist_ok=True)
            os.makedirs(temp_depth_dir, exist_ok=True)

            # Copy frames to temporary directories
            for i, (rgb_path, depth_path) in enumerate(zip(frame_paths, depth_paths)):
                if os.path.exists(rgb_path):
                    shutil.copy(rgb_path, os.path.join(temp_rgb_dir, f'image_{i}.jpg'))
                if os.path.exists(depth_path):
                    shutil.copy(depth_path, os.path.join(temp_depth_dir, f'image_{i}.npy'))

            # Create zip archives for RGB and depth images
            rgb_zip_path = os.path.join(recording_dir, 'rgb_images_collection.zip')
            depth_zip_path = os.path.join(recording_dir, 'depth_images_collection.zip')
            create_zip_archive(temp_rgb_dir, rgb_zip_path)
            create_zip_archive(temp_depth_dir, depth_zip_path)

            # Hit the Hamer API with the newly generated zips
            response_encoded = get_base64_encoded_hamer_response(rgb_zip_path, depth_zip_path)
            if response_encoded:
                csv_path = base64_to_csv_path(response_encoded, f'{recording_dir}/predictions_hamer.csv')
                return csv_path
            
            return None

    # ========================================================
    # 2) transform_coordinates & coordinate system transforms
    # ========================================================
    X = np.array([
        [0.068, -0.986, 0.152, -0.108],
        [0.998, 0.065, -0.023, 0.0],
        [0.013, 0.153, 0.988, -0.044],
        [0.0, 0.0, 0.0, 1.0]
    ])

    Y = np.array([
        [-0.47, 0.587, -0.659, 0.73929],
        [0.877, 0.392, -0.276, -0.16997],
        [0.096, -0.708, -0.7, 0.86356],
        [0.0, 0.0, 0.0, 1.0]
    ])

    def transform_coordinates(x, y, z):
        """Transforms coordinates from input space to the cobot base."""
        B = np.eye(4)
        B[:3, 3] = [x / 1000, y / 1000, z / 1000]  # mm --> meters
        A = Y @ B @ np.linalg.inv(X)
        # Convert back to mm
        transformed_x, transformed_y, transformed_z = A[:3, 3] * 1000
        return transformed_x, transformed_y, transformed_z

    # ===============================================
    # 3) initial_frame_centers for object detection
    # ===============================================
    class ObjectDetector:
        """
        Example stub for your real Gemini-based object detector. 
        In reality, you'd import from gemini_oop_object_detection.
        """
        def __init__(self, api_key, recording_dir):
            self.api_key = api_key
            self.recording_dir = recording_dir

        def get_object_center(self, rgb_image, object_name):
            # In practice, you might call the Gemini API here.
            # Let's just pretend we found a bounding box center at (100, 100).
            center_x, center_y = 100, 100
            box = [90, 90, 110, 110]  # example bounding box
            confidence = 0.9
            return center_x, center_y, box, confidence

    def initial_frame_centers(recording_dir, classes):
        """
        Passes the 0th frame (or a chosen frame index) to the Gemini API 
        and returns the x,y center point values of the objects. Then
        uses depth to transform the coordinates to the cobot base.
        """
        gemini_model = ObjectDetector(api_key=gemini_api_key, recording_dir=recording_dir)
        object_centers = {}

        # Hard-coded example to read frame #10 in your original code
        # Adjust to your preference
        rgb_image_path = f"{recording_dir}/rgb_images_data_collection/image_10.jpg"
        depth_image_path = f"{recording_dir}/depth_images_data_collection/image_10.npy"

        if not os.path.exists(rgb_image_path) or not os.path.exists(depth_image_path):
            print("Warning: Could not find default image_10 for detection.")
            return {}

        # Load the images
        rgb_image = Image.open(rgb_image_path)
        depth_image = np.load(depth_image_path)

        # Setup realsense intrinsics
        intrinsics = rs.intrinsics()
        intrinsics.width = 640
        intrinsics.height = 480
        intrinsics.ppx = 329.1317443847656
        intrinsics.ppy = 240.29669189453125
        intrinsics.fx = 611.084594726562
        intrinsics.fy = 609.7639770507812
        intrinsics.model = rs.distortion.inverse_brown_conrady
        intrinsics.coeffs = [0, 0, 0, 0, 0]

        # Get center for each object class
        for name in classes:
            center_x, center_y, box, confidence = gemini_model.get_object_center(rgb_image, name)
            point_3d = deproject_pixel_to_point(depth_image, (center_x, center_y), intrinsics)
            transformed_x, transformed_y, transformed_z = transform_coordinates(point_3d[0], point_3d[1], point_3d[2])
            object_centers[name] = [transformed_x, transformed_y, transformed_z]

        return object_centers

    # ===================================
    # 4) post_process_csv implementation
    # ===================================
    def post_process_csv(input_csv_path, output_csv_path, input_dict):
        """
        Takes an original CSV file (predictions_hamer.csv), reads in the data,
        then writes out a single row CSV containing the points in columns:
        'P0_X', 'P0_Y', 'P0_Z', 'P0_C', etc., plus the new 'Input_0_x,y,z' and 'Input_1_x,y,z'.
        """
        with open(input_csv_path, mode='r') as infile, open(output_csv_path, mode='w', newline='') as outfile:
            reader = csv.DictReader(infile)
            # Customize these fieldnames as needed
            fieldnames = ['Input_0_x', 'Input_0_y', 'Input_0_z', 'Input_1_x', 'Input_1_y', 'Input_1_z']
            
            # We assume up to 81 points (P0..P80)
            for i in range(81):
                fieldnames.extend([f'P{i}_X', f'P{i}_Y', f'P{i}_Z', f'P{i}_C'])
            
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()
            
            # Initialize row_data
            row_data = {field: 0 for field in fieldnames}
            
            # Setting input values from the dictionary:
            # Adjust keys if your dictionary keys differ
            # For example, if "white glass cup" is "Input_0" and "red soda can" is "Input_1":
            row_data['Input_0_x'], row_data['Input_0_y'], row_data['Input_0_z'] = input_dict.get('white glass cup', [0,0,0])
            row_data['Input_1_x'], row_data['Input_1_y'], row_data['Input_1_z'] = input_dict.get('red soda can', [0,0,0])
            
            # Fill in the P{i}_X/Y/Z/C from the existing CSV
            for i, row in enumerate(reader):
                row_data[f'P{i}_X'] = row.get('X', 0)
                row_data[f'P{i}_Y'] = row.get('Y', 0)
                row_data[f'P{i}_Z'] = row.get('Z', 0)
                row_data[f'P{i}_C'] = row.get('C', 0)
            
            # Write just one row (the combined row) to the CSV
            writer.writerow(row_data)

    # ============================
    # Main Execution Pipeline
    # ============================

    # 1) Extract frames & call Hamer API
    csv_path = collection_d(recording_dir, start_time, end_time, fps)
    if csv_path is None:
        print("No CSV path returned from collection_d. Exiting.")
        return None

    # 2) Compute initial frame centers
    input_dict = initial_frame_centers(recording_dir, classes)

    # 3) Post process the CSV with input_dict
    post_process_csv(csv_path, output_csv_path, input_dict)
    print(f"Processed CSV saved at: {output_csv_path}")

    return output_csv_path

def ellm(recording_dir): 
    url = "https://dev-egpt.techo.camp/predict"
    headers = {
        "Content-Type": "application/json"
    }
    #TODO: make the payload customizable.
    # enable change in video links, agents, etc.
    payload = None
    # Load the payload from a JSON file
    with open("payload.json", "r") as file:
        payload = json.load(file)

    print(f"================ PAYLOAD ================ +\n{payload['question']}\n================ PAYLOAD ================")
    analyzer = VideoAnalyzer(payload=payload)
    analyzer.upload_video_to_bucket("test_1.mp4",f"{recording_dir}/color.mp4")
    response = analyzer.get_ellm_response()
    print(response)
    return response

# ======================
# Example usage
# ======================
# if __name__ == "__main__":
#     recording_dir = "recordings/Recorded_Demo"
#     fps = 10
#     classes = ["red soda can", "white glass cup"]
#     action = "pouring"

#     # Where your raw and processed CSV will reside
#     input_csv_path = f"{recording_dir}/predictions_hamer.csv"         # This will be created by collection_d
#     output_csv_path = f"{recording_dir}/processed_predictions_hamer.csv"

#     # Example Gemini API key
#     gemini_api_key = GEMINI_API_KEY

#     # Optionally, an ELLM response to be passed in if you want to handle multiple actions:

#     ellm_response = ellm()
#     # Extract start_time and end_time for the "pouring" action from ellm_response
#     pouring_action = ellm_response.get(action, [])
#     if pouring_action:
#         start_time_str = pouring_action[0]["start_time"]
#         end_time_str = pouring_action[0]["end_time"]
#         start_time = int(start_time_str.split(":")[0]) * 60 + int(start_time_str.split(":")[1])
#         end_time = int(end_time_str.split(":")[0]) * 60 + int(end_time_str.split(":")[1])
#     else:
#         print("No pouring action found in ellm_response.")
    
#     # Sample Response from ELLM
#     # ellm_response = {
#     #     "overall_task_name": "picking up and placing a can",
#     #     "objects": ["can", "small black box"],
#     #     "picking up": [
#     #         {
#     #             "start_time": "00:02",
#     #             "end_time": "00:05",
#     #             "object_name": "can",
#     #             "notes": "Human reaches for and grasps the can.",
#     #             "action_timestamp": "00:03"
#     #         },
#     #     ],
#     #     "placing": [
#     #         {
#     #             "start_time": "00:05",
#     #             "end_time": "00:07",
#     #             "object_name": "can",
#     #             "notes": "Human moves the can and places it down on the table.",
#     #             "action_timestamp": "00:07"
#     #         }
#     #     ]
#     # }

#     processed_csv = combined_pipeline(
#         recording_dir=recording_dir,
#         start_time=start_time,
#         end_time=end_time,
#         fps=fps,
#         classes=classes,
#         input_csv_path=input_csv_path,
#         output_csv_path=output_csv_path,
#         gemini_api_key=gemini_api_key,
#         ellm_response=ellm_response
#     )


SHAPE OF COLOR FRAMES: 76
Processed CSV saved at: recordings/Recorded_Demo/processed_predictions_hamer.csv


In [ ]:
import os
import json
from object_detection.vdeo_analysis_ellm_sudio import VideoAnalyzer

# ----------------------------------------------------------------------------------------
# Example: Loop through all subfolders like 'recording_1', 'recording_2', 'recording_3', etc.
# ----------------------------------------------------------------------------------------

def process_all_recordings(base_recordings_dir, action="pouring", classes=None, fps=10, gemini_api_key=""):
    """
    Loop over all subfolders (e.g., 'recording_1', 'recording_2') in base_recordings_dir.
    For each subfolder:
       1) Run ellm() to get LLM response
       2) Extract start_time & end_time from that response
       3) Invoke combined_pipeline to produce a processed CSV
    """
    if classes is None:
        classes = ["red soda can", "white glass cup"]  # or whatever default classes you want
    
    # Get list of all subfolders
    for folder_name in os.listdir(base_recordings_dir):
        subfolder_path = os.path.join(base_recordings_dir, folder_name)
        
        # Only process if it's a directory and frames.h5 is present
        if not os.path.isdir(subfolder_path):
            continue
        
        h5_file_path = os.path.join(subfolder_path, "frames.h5")
        if not os.path.exists(h5_file_path):
            print(f"Skipping {subfolder_path}: frames.h5 not found.")
            continue
        
        print(f"\n====== Processing {subfolder_path} ======")
        
        # 1) Call ELLM for that recording directory
        try:
            ellm_response = ellm(subfolder_path)  # Your ellm() function
        except Exception as e:
            print(f"Warning: ELLM request failed for {subfolder_path}, skipping.\nError: {e}")
            continue
        
        # 2) Extract start_time & end_time from ELLM response for the desired action
        #    This part depends on your actual ELLM response structure
        #    We'll assume it's the same pattern as your example
        action_data = ellm_response.get(action, [])
        if not action_data:
            print(f"No '{action}' action found in ELLM response for {subfolder_path}. Skipping.")
            continue
        
        # Extract first item’s start/end times in MM:SS format
        try:
            start_time_str = action_data[0]["start_time"]  # e.g. "00:02"
            end_time_str = action_data[0]["end_time"]      # e.g. "00:05"
            start_time = int(start_time_str.split(":")[0]) * 60 + int(start_time_str.split(":")[1])
            end_time = int(end_time_str.split(":")[0]) * 60 + int(end_time_str.split(":")[1])
        except Exception as e:
            print(f"Could not parse start/end time for action '{action}' in {subfolder_path}: {e}")
            continue
        
        # 3) Set up input/ output CSV paths for that subfolder
        input_csv_path = os.path.join(subfolder_path, "predictions_hamer.csv")
        output_csv_path = os.path.join(subfolder_path, "processed_predictions_hamer.csv")
        
        # 4) Run the combined pipeline
        try:
            processed_csv = combined_pipeline(
                recording_dir=subfolder_path,
                start_time=start_time,
                end_time=end_time,
                fps=fps,
                classes=classes,
                input_csv_path=input_csv_path,
                output_csv_path=output_csv_path,
                gemini_api_key=gemini_api_key,
                ellm_response=ellm_response  # optional
            )
            print(f"Processed CSV for {folder_name} = {processed_csv}")
        except Exception as e:
            print(f"Error running combined_pipeline for {subfolder_path}:\n{e}")


if __name__ == "__main__":
    # Path containing multiple subfolders: recording_1, recording_2, ...
    base_recordings_dir = "recordings"
    
    # Example Gemini API key
    gemini_api_key = GEMINI_API_KEY
    
    # Classes you want to detect
    classes = ["red soda can", "white glass cup"]
    
    # Now call our processing function
    process_all_recordings(
        base_recordings_dir=base_recordings_dir,
        action="pouring",
        classes=classes,
        fps=10,
        gemini_api_key=gemini_api_key
    )
